## Patrón 1 — LCEL-like con pipes (si la versión lo permite)

Este patrón usa el operador | para encadenar componentes:

Aquí cada paso pasa su salida al siguiente:

prompt → LLM → limpieza

Muy legible y conciso

In [1]:
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda

# 1) Instanciar LLM liviano
llm = OllamaLLM(model="tinyllama:latest", temperature=0.2, num_ctx=1024)

# 2) Prompt sencillo
prompt = PromptTemplate.from_template("Tell about ancient cultures in {topic}.")

# 3) Runnable para limpiar texto
def trim(text: str) -> str:
    return text.strip() # elimina espacios al inicio y final

trim_runnable = RunnableLambda(trim)

# 4) Cadena con pipes
pipeline = prompt | llm | trim_runnable

# 5) Ejecutar
result = pipeline.invoke({"topic": "Lima"})
print("Response:", result)


Fun fact: Lima, the capital city of Peru, has a rich history dating back to pre-Columbian times. Here are some notable ancient cultures that you might be interested in learning more about:

1. Inca Empire: The Incas were a powerful empire that ruled over parts of South America for over 200 years. They built impressive structures such as the Sacsayhuaman Fortress, which is made up of massive stone blocks arranged in a stunning pattern.

2. Moche Culture: This ancient civilization was based in the coastal region of Peru and is known for their intricate carvings on rock formations. They also built impressive tombs and temples that are still standing today.

3. Nazca Lines: The Nazca Lines are a series of geoglyphs that were created by the Nazca people in the 2nd millennium BC. They depict animals, birds, and other symbols that have fascinated scientists and historians alike.

4. Chavín de Apamayo: This ancient civilization was based in the Andes Mountains of Peru and is known for their in

In [2]:
result = pipeline.invoke({"topic": "Sao Paulo Brasil"})
print("Response:", result)

Response: Sao Paulo, the largest city in Brazil, has a rich and diverse cultural heritage that dates back to pre-Columbian times. Here are some of the most notable ancient cultures that have left their mark on the city:

1. Aztecs - The Aztecs were a powerful civilization that ruled over much of present-day Mexico from 1325 to 1521. Their influence can still be seen in the city's architecture, including the famous Templo de Debod temple complex, which was built in the early 20th century and moved to its current location in 1967.

2. Incas - The Incas were a powerful empire that ruled over parts of Peru from around 1438 to 1532. Their influence can still be seen in the city's architecture, including the impressive Inca ruins at Machu Picchu, which are located just outside of the city.

3. Guarani - The Guarani people were a tribe that lived in the region around present-day Sao Paulo for thousands of years before being conquered by the Spanish in the 16th century. Their influence can sti

## Patrón 2 — Múltiples chains clásicos encadenados

In [4]:
#from langchain_ollama import OllamaLLM
#from langchain_core.prompts import PromptTemplate

# 1) Modelo pequeño
#llm = OllamaLLM(model="tinyllama:latest", temperature=0.2, num_ctx=1024)

# ----------------------------------------
# Chain 1 — Resumen
# ----------------------------------------
prompt_summary = PromptTemplate(
    input_variables=["text"],
    template="Summarize this text in one short sentence:\n\n{text}"
)

chain_summary = prompt_summary | llm

# ----------------------------------------
# Chain 2 — Clasificación
# ----------------------------------------
prompt_classify = PromptTemplate(
    input_variables=["summary"],
    template=(
        "Given this summary:\n"
        "{summary}\n"
        "Classify it as 'simple' or 'complex'."
    )
)

chain_classify = prompt_classify | llm

# ----------------------------------------
# Encadenar
# ----------------------------------------
def summarize_and_classify(text: str):
    # Paso 1: resumen
    resumen = chain_summary.invoke({"text": text})
    # Paso 2: clasificación
    categoria = chain_classify.invoke({"summary": resumen})
    return resumen, categoria

# ----------------------------------------
# Ejemplo de uso
# ----------------------------------------
input_text = "Deep learning models can overfit small datasets unless regularized."
summary, label = summarize_and_classify(input_text)

print("Resumen:", summary)
print("Clasificación:", label)

Resumen: Regularization techniques can help prevent deep learning models from overfitting on small datasets, ensuring better generalization capabilities and improved performance on larger datasets.
Clasificación: The article discusses the significance of regularization techniques in deep learning models for preventing overfitting on small datasets and improving their generalization capabilities and performance on larger datasets. The classification is 'simple' since the article does not delve into complex or advanced aspects of the topic.
